**Kaggle:** https://www.kaggle.com/competitions/neoai-2025-cluster-pictures/data

In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.cluster import KMeans, SpectralClustering, MiniBatchKMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import timm
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torchvision.transforms as transforms

device = torch.device('cpu')
PATH = f'{os.getcwd()}/cluster_pictures'


class EmbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model("tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=True, num_classes=0)

    def forward(self, image):
        x = self.model(image)
        return x

In [32]:
def generate_submit(pred_cluster):
    sub = pd.DataFrame()
    sub['id'] = np.arange(len(pred_cluster))
    sub['target'] = pred_cluster
    print(sub.head(10))
    sub.to_csv(f'{PATH}/submission.csv', index=False)


X_1 = np.load(f'{PATH}/data_1.npz')
X_1 = X_1.f.arr_0
X_2 = np.load(f'{PATH}/data_2.npz')
X_2 = X_2.f.arr_0
X_1 = X_1.transpose(0, 2, 1)

In [33]:
X_vec = np.concatenate([X_1.reshape(3840, 512), X_2.reshape(3840, 512)], axis=1)
pd.DataFrame(X_vec)

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,0.195612,0.364533,0.534417,0.756469,0.901142,0.907384,1.049618,1.324492,1.518135,1.803956,...,46.100075,45.004734,44.615540,44.115616,42.223782,39.451073,38.294128,38.495472,35.695091,3.597008e+01
1,0.045011,0.043794,0.031720,0.077549,0.157818,0.230448,0.302324,0.409228,0.536734,0.800696,...,0.000414,0.000645,0.000223,0.000246,0.000137,0.001611,0.000150,0.000022,0.000203,7.383991e-05
2,0.395572,0.400565,0.493983,0.781316,0.816927,0.795136,0.717517,0.824353,0.819369,0.830072,...,0.035554,0.028216,0.020231,0.035994,0.005459,0.041972,0.022221,0.005439,0.009930,8.029810e-03
3,2.361179,2.339077,2.265174,2.208350,1.990410,1.825174,1.431634,1.046068,0.596698,0.439571,...,23.939562,27.247726,30.924366,31.343538,35.371971,37.934700,40.588692,41.043694,43.051456,4.105296e+01
4,0.383065,0.376546,0.738897,0.621795,0.619263,0.481274,0.423059,0.415207,0.381153,0.357684,...,33.833366,34.554569,35.129917,35.875984,34.055073,35.818428,36.012753,35.413849,37.665771,3.696240e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3835,1.108132,0.918888,0.797333,0.870280,0.980816,1.122145,1.161255,1.204587,1.056555,1.060452,...,61.354294,65.442551,64.481613,55.927208,42.655872,52.729092,52.329456,45.066368,49.252895,4.650184e+01
3836,2.140607,2.225540,2.267492,2.265918,2.234048,2.194899,2.144015,2.082687,2.014283,1.927377,...,1.333659,0.384417,0.134799,0.034123,0.000847,0.000687,0.000062,0.000004,0.000001,7.250301e-08
3837,0.339273,0.306973,0.228924,0.250321,0.265130,0.302217,0.305875,0.328013,0.345345,0.392523,...,33.915546,35.998436,34.188660,31.977402,29.118612,28.448759,29.991125,29.725077,30.519299,2.973379e+01
3838,1.220243,1.428414,1.682581,1.928777,1.812571,1.570408,1.267960,1.109619,0.911661,0.809632,...,4.275742,4.657784,4.466135,3.624427,1.429203,1.536545,0.786550,0.228315,0.308110,4.121899e-01


In [34]:
X_vec_sc = StandardScaler().fit_transform(X_vec)
X_vec_pca = PCA(n_components=128).fit_transform(X_vec_sc)
X_vec_pca = np.asarray(X_vec_pca, dtype=np.float64)

pred_cluster = KMeans(n_clusters=32).fit_predict(X_vec_pca)
print('silhouette_score', silhouette_score(X_vec, pred_cluster))
print('davies_bouldin_score', davies_bouldin_score(X_vec, pred_cluster))
print('calinski_harabasz_score', calinski_harabasz_score(X_vec, pred_cluster))
# generate_submit(pred_cluster)

silhouette_score -0.007028840947896242
davies_bouldin_score 4.6607044059259914
calinski_harabasz_score 57.599063873291016


In [35]:
X_full = np.stack([X_1, X_2, X_1 - X_2], axis=1)

X_full_t = torch.FloatTensor(X_full).to(device)
X_full_t = F.interpolate(
    X_full_t,
    size=(32, 224),
    mode='bilinear',
    align_corners=False
)
X_full_t.shape

torch.Size([3840, 3, 32, 224])

In [36]:
transform = transforms.Compose([
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.225, 0.225, 0.225]
    )
])
X_loader = DataLoader(transform(X_full_t), batch_size=32, shuffle=False)

In [37]:
embnet = EmbNet().to(device)
embnet.eval()

embeddings = []
with torch.no_grad():
    for X_batch in X_loader:
        emb = embnet(X_batch)
        embeddings.append(emb.cpu())

embeddings = torch.cat(embeddings, dim=0).cpu().numpy()
print(embeddings.shape)
print(embeddings)

(3840, 320)
[[-0.08223662  0.21514215  0.12077173 ...  0.12423897 -0.03953214
   0.32238334]
 [-0.09487666  0.22416924  0.16958046 ... -0.01351507 -0.14695527
   0.28225714]
 [-0.09058818  0.21059348  0.18069062 ...  0.0280267  -0.14502785
   0.2927181 ]
 ...
 [-0.09480704  0.2183027   0.14935905 ...  0.0456952  -0.09372184
   0.31380272]
 [-0.08888367  0.21797302  0.17661996 ...  0.01129002 -0.1257107
   0.29541528]
 [-0.07424718  0.22536552  0.1565906  ...  0.05871019 -0.01734867
   0.30152225]]


In [ ]:
pred_cluster = KMeans(n_clusters=32, random_state=42).fit_predict(embeddings)
print('silhouette_score', silhouette_score(embeddings, pred_cluster))
print('davies_bouldin_score', davies_bouldin_score(embeddings, pred_cluster))
print('calinski_harabasz_score', calinski_harabasz_score(embeddings, pred_cluster))
# generate_submit(pred_cluster)

silhouette_score 0.1469872146844864
davies_bouldin_score 1.5904948694176668
calinski_harabasz_score 914.5587158203125
   id  target
0   0      29
1   1      14
2   2      21
3   3      19
4   4       0
5   5      14
6   6       4
7   7       1
8   8       0
9   9      24


In [39]:
pred_cluster_other = KMeans(n_clusters=32, random_state=25).fit_predict(embeddings)
print('silhouette_score', silhouette_score(embeddings, pred_cluster_other))
print('davies_bouldin_score', davies_bouldin_score(embeddings, pred_cluster_other))
print('calinski_harabasz_score', calinski_harabasz_score(embeddings, pred_cluster_other))

silhouette_score 0.1440155953168869
davies_bouldin_score 1.5540286105464969
calinski_harabasz_score 920.762451171875


In [40]:
print(adjusted_rand_score(pred_cluster, pred_cluster_other))
print(normalized_mutual_info_score(pred_cluster, pred_cluster_other))
print(adjusted_mutual_info_score(pred_cluster, pred_cluster_other))

0.4631432831212659
0.7266874380263112
0.7151584272848057
